In [ ]:
Project: covid19datahub_covid19dev
    - # Stars: 252
    - # Forks: 92
    - Last Commit: Jun 30, 2025

Project: echonet_dynamic
- # Stars: 164
- # Forks: 48
- Last Commit: May 1, 2026

Project: opengeospatial_ogc-geosparql
- # Stars: 105
- # Forks: 21
- Last Commit: Sep 21, 2026

Project: egri-nagy_sgpdec
- # Stars: 22
- # Forks: 3
- Last Commit: Jun 30, 2026

Project: eendebakpt_oapackage
- # Stars: 39
- # Forks: 10
- Last Commit: Feb 4, 2026

Project: wojdyr_fityk
- # Stars: 306
- # Forks: 61
- Last Commit: Mar 2, 2026

Project: smearle_gym-city
- # Stars: 161
- # Forks: 24
- Last Commit: Apr 4, 2025

Project: acclab_dabestr
- # Stars: 230
- # Forks: 38
- Last Commit: Jul 29, 2026

Project: aol_moloch
- # Stars: 7,509
- # Forks: 1,168
- Last Commit: Sep 28, 2026

Project: ncoudray_deeppath
- # Stars: 537
- # Forks: 214
- Last Commit: Sep 24, 2024

In [ ]:
!python -m pip install -U python-woc pandas matplotlib

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
import time

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
# if you got an API key (faster, fewer rate limits):
# woc = WocMapsRemote(base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY")

projects = [
    "covid19datahub_covid19dev", "echonet_dynamic", "opengeospatial_ogc-geosparql",
    "egri-nagy_sgpdec", "eendebakpt_oapackage", "wojdyr_fityk", "smearle_gym-city",
    "acclab_dabestr", "aol_moloch", "ncoudray_deeppath",
]

list_df_commits = []
for prj in projects:
    prj = prj.lower().replace('/', '_', 2)
    commits = woc.get_values('p2c', prj)
    print(prj, len(commits))
    d = pd.DataFrame(commits, columns=["sha1"])
    d['project'] = prj
    list_df_commits.append(d)

df = pd.concat(list_df_commits, ignore_index=True)
df.to_csv('df_commits.csv', index=False)
df.head(1)

covid19datahub_covid19dev 8683
echonet_dynamic 178
opengeospatial_ogc-geosparql 4049
egri-nagy_sgpdec 2956
eendebakpt_oapackage 2077
wojdyr_fityk 1849
smearle_gym-city 459
acclab_dabestr 736
aol_moloch 12829
ncoudray_deeppath 552


,sha1,project
0,000082696d0d42a74a4df629f25c16b853a62873,covid19datahub_covid19dev


In [3]:
chunks = [df['sha1'][x:x+10].to_list() for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks):
    for attempt in range(5):
        try:
            res, err = woc.get_values_many('commit.tch', chunk)
            break
        except Exception:
            time.sleep(60)   # rate limited: wait and retry
            woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
    else:
        print("skipped a chunk after 5 failures")
        continue

    res = {k: v[0] for k, v in res.items()}
    time.sleep(1)
    if err:
        print('Got Errors', err)

    for commit_sha, commit in res.items():
        commit_data.append({
            'commit': commit_sha,
            'tree': commit[0],
            'parent': list(commit[1]),
            'author': commit[2][0],
            'author_time': int(commit[2][1]),
            'author_tz': commit[2][2],
            'committer': commit[3][0],
            'committer_time': int(commit[3][1]),
            'committer_tz': commit[3][2],
            'message': commit[4],
        })

df_commit_data = pd.DataFrame(commit_data).merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 42%|██████████████████████████████████████████████████████████████████████▍                                                                                                  | 1433/3437 [24:50<34:59,  1.05s/it]

Got Errors {'7860edd766f44589912be7b247a2ec4240cef9cb': 'Key 7860edd766f44589912be7b247a2ec4240cef9cb not found in /da5_fast/All.sha1c/commit_120.tch'}


 49%|███████████████████████████████████████████████████████████████████████████████████▌                                                                                     | 1700/3437 [29:28<30:04,  1.04s/it]

Got Errors {'8a84895e840134b972ae07e52f0655c5ff1a5437': 'Key 8a84895e840134b972ae07e52f0655c5ff1a5437 not found in /da5_fast/All.sha1c/commit_10.tch'}


 50%|███████████████████████████████████████████████████████████████████████████████████▉                                                                                     | 1708/3437 [29:37<30:04,  1.04s/it]

...

Got Errors {'5ad63643d785731871135980519b49027a68ccd0': 'Key 5ad63643d785731871135980519b49027a68ccd0 not found in /da5_fast/All.sha1c/commit_90.tch'}


 80%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                 | 2761/3437 [47:55<11:43,  1.04s/it]

Got Errors {'8467f712c58d948bf88a13e347ff849ea6d32205': 'Key 8467f712c58d948bf88a13e347ff849ea6d32205 not found in /da5_fast/All.sha1c/commit_4.tch'}


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3437/3437 [59:38<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,000082696d0d42a74a4df629f25c16b853a62873,21fd1d9f8555522535f187723b6436fe8cf6b9ba,[a0d0412d5b4f858e41a33011f90753e157cb497c],Emanuele Guidotti <emanuele.guidotti@unine.ch>,1687917959,+0000,Emanuele Guidotti <emanuele.guidotti@unine.ch>,1687917959,+0000,Built site for COVID19: 3.0.0@ec8865d\n,000082696d0d42a74a4df629f25c16b853a62873,covid19datahub_covid19dev
1,00031ee2f9d57f1a050ce76bf43796c4d1124a55,95a34e5a0469a7fa12d4ef99cbf92f20eb92d74e,[74eea47b24a9e76bc3463d279a94af7da34cb861],Emanuele Guidotti <emanuele.guidotti@unine.ch>,1620625317,+0000,Emanuele Guidotti <emanuele.guidotti@unine.ch>,1620625317,+0000,Built site for COVID19: 2.3.0@f6b203b\n,00031ee2f9d57f1a050ce76bf43796c4d1124a55,covid19datahub_covid19dev


In [4]:
summary = df_commit_data.groupby('project').agg(
    num_commits=('commit', 'nunique'),
    num_authors=('author', 'nunique'),
    max_time=('author_time', 'max'),
    min_time=('author_time', 'min'),
)
summary

,num_commits,num_authors,max_time,min_time
project,,,,
acclab_dabestr,736,28,1757925990,1537494537
aol_moloch,12825,292,1762207138,1341591146
covid19datahub_covid19dev,8683,35,1762348316,1585010715
echonet_dynamic,178,9,1742018801,1569438238
eendebakpt_oapackage,2072,17,1737929025,1422542337
egri-nagy_sgpdec,2955,19,1762296673,1334806503
ncoudray_deeppath,552,24,1732552722,1495553134
opengeospatial_ogc-geosparql,4049,44,1761225811,1602232413
smearle_gym-city,459,12,1746496404,1534818586


In [5]:
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].rename(
    columns={'author_time': 'time'})

yournetid = 'lsantucc'
dfinf.to_csv(yournetid + '_project_summary.csv', index=False, sep=';', mode='a', header=False)
dfinf.head(1)

,project,commit,author,time,message
0,covid19datahub_covid19dev,000082696d0d42a74a4df629f25c16b853a62873,Emanuele Guidotti <emanuele.guidotti@unine.ch>,1687917959,Built site for COVID19: 3.0.0@ec8865d\n


In [ ]:
## Project summary (WoC commits + GitHub)

| Project | WoC commits | WoC authors | Max time | Min time | GH stars | GH forks | GH last commit |
|---|---|---|---|---|---|---|---|
| acclab_dabestr | 736 | 28 | 1757925990 (Sep 15, 2025) | 1537494537 (Sep 21, 2018) | 230 | 38 | Jul 29, 2026 |
| aol_moloch | 12825 | 292 | 1762207138 (Nov 3, 2025) | 1341591146 (Jul 6, 2012) | 7,509 | 1,168 | Sep 28, 2026 |
| covid19datahub_covid19dev | 8683 | 35 | 1762348316 (Nov 5, 2025) | 1585010715 (Mar 24, 2020) | 252 | 92 | Jun 30, 2025 |
| echonet_dynamic | 178 | 9 | 1742018801 (Mar 15, 2025) | 1569438238 (Sep 25, 2019) | 164 | 48 | May 1, 2026 |
| eendebakpt_oapackage | 2072 | 17 | 1737929025 (Jan 26, 2025) | 1422542337 (Jan 29, 2015) | 39 | 10 | Feb 4, 2026 |
| egri-nagy_sgpdec | 2955 | 19 | 1762296673 (Nov 4, 2025) | 1334806503 (Apr 19, 2012) | 22 | 3 | Jun 30, 2026 |
| ncoudray_deeppath | 552 | 24 | 1732552722 (Nov 25, 2024) | 1495553134 (May 23, 2017) | 537 | 214 | Sep 24, 2024 |
| opengeospatial_ogc-geosparql | 4049 | 44 | 1761225811 (Oct 23, 2025) | 1602232413 (Oct 9, 2020) | 105 | 21 | Sep 21, 2026 |
| smearle_gym-city | 459 | 12 | 1746496404 (May 6, 2025) | 1534818586 (Aug 21, 2018) | 161 | 24 | Apr 4, 2025 |
| wojdyr_fityk | 1840 | 21 | 1738220501 (Jan 30, 2025) | 1055880472 (Jun 17, 2003) | 306 | 61 | Mar 2, 2026 |

Times are Unix timestamps of author time (UTC date in parentheses). GitHub data collected Sep 28, 2026.
On GitHub, aol/moloch now redirects to arkime/arkime and egri-nagy/sgpdec to gap-packages/sgpdec.
The WoC data ends around Nov 2025 because that is when its snapshot was taken, so it does not include later GitHub commits.